# Belgian Urban Heat AI — CNN GPU Training (Google Colab)

**Purpose:** Train the `HeatCNN` on Sentinel-2 image patches using a free Colab T4 GPU.  
100 epochs on T4 ≈ 2 minutes. Same model on CPU ≈ 40 minutes.

## Before running
1. **Runtime → Change runtime type → T4 GPU → Save**
2. Upload the 6 `.npy` files from `data/processed/` to **Google Drive** at:
   ```
   My Drive/urban-heat-ai/data/
   ```
   Files to upload:
   - `X_patches.npy` — Sentinel-2 image patches (CNN input)
   - `y_patches.npy` — anomaly targets for patches
   - `X_train.npy`, `X_test.npy` — scaled tabular features
   - `y_train.npy`, `y_test.npy` — anomaly targets for tabular models
3. Run all cells in order.

In [ ]:
# ── 1. Verify GPU ─────────────────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "No GPU found! Go to Runtime > Change runtime type > T4 GPU"
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
DEVICE = 'cuda'

In [ ]:
# ── 2. Mount Google Drive ──────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DATA = '/content/drive/MyDrive/urban-heat-ai/data/'

import os
files = os.listdir(DATA)
print('Files found in Drive folder:')
for f in sorted(files):
    size = os.path.getsize(DATA + f) / 1024
    print(f'  {f:<25} {size:.0f} KB')

In [ ]:
# ── 3. Imports ─────────────────────────────────────────────────────────────────
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
print('Imports OK')

In [ ]:
# ── 4. Load patches from Drive ─────────────────────────────────────────────────
X_patches = np.load(DATA + 'X_patches.npy').astype('float32')
y_patches = np.load(DATA + 'y_patches.npy').astype('float32')

print(f'X_patches shape: {X_patches.shape}  (samples, channels, H, W)')
print(f'y_patches shape: {y_patches.shape}')
print(f'Anomaly range:   {y_patches.min():.2f} to {y_patches.max():.2f} degC')

PATCH_SIZE = X_patches.shape[-1]  # 16
IN_CHANNELS = X_patches.shape[1]  # 3 (B04, B08, B11)
print(f'Patch size: {PATCH_SIZE}x{PATCH_SIZE}, channels: {IN_CHANNELS}')

In [ ]:
# ── 5. Train / test split ──────────────────────────────────────────────────────
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(
    X_patches, y_patches, test_size=0.2, random_state=42
)

train_loader = DataLoader(
    TensorDataset(torch.from_numpy(Xp_tr), torch.from_numpy(yp_tr)),
    batch_size=64, shuffle=True, num_workers=2, pin_memory=True
)

print(f'Train: {len(Xp_tr)} patches | Test: {len(Xp_te)} patches')
print(f'Batches per epoch: {len(train_loader)}')

In [ ]:
# ── 6. Model definition (identical to local notebook) ─────────────────────────
class HeatCNN(nn.Module):
    def __init__(self, in_channels=IN_CHANNELS, patch_size=PATCH_SIZE):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1),          nn.ReLU(), nn.MaxPool2d(2),
        )
        flat = 32 * (patch_size // 4) ** 2
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(flat, 64), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(64, 1)
        )
    def forward(self, x):
        return self.head(self.conv(x)).squeeze(1)

cnn_model = HeatCNN().to(DEVICE)
total_params = sum(p.numel() for p in cnn_model.parameters())
print(f'HeatCNN on {DEVICE} | Parameters: {total_params:,}')

In [ ]:
# ── 7. Train — 100 epochs on GPU ───────────────────────────────────────────────
EPOCHS = 100

optimizer = optim.Adam(cnn_model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)
criterion = nn.MSELoss()

loss_history = []
cnn_model.train()

for epoch in range(1, EPOCHS + 1):
    epoch_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(cnn_model(xb), yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / len(train_loader)
    loss_history.append(avg_loss)
    if epoch % 10 == 0 or epoch == 1:
        print(f'Epoch {epoch:03d}/{EPOCHS}  loss: {avg_loss:.4f}')

print('\nTraining complete.')

In [ ]:
# ── 8. Evaluate ────────────────────────────────────────────────────────────────
cnn_model.eval()
with torch.no_grad():
    preds = cnn_model(
        torch.from_numpy(Xp_te).to(DEVICE)
    ).cpu().numpy()

cnn_rmse = np.sqrt(mean_squared_error(yp_te, preds))
cnn_mae  = mean_absolute_error(yp_te, preds)
cnn_r2   = r2_score(yp_te, preds)

print('=' * 45)
print(f'CNN (100 epochs, T4 GPU)')
print(f'  RMSE : {cnn_rmse:.4f} degC')
print(f'  MAE  : {cnn_mae:.4f} degC')
print(f'  R2   : {cnn_r2:.4f}')
print('=' * 45)
print()
print('v2 comparison (local, 20 epochs CPU):')
print('  RMSE : 0.5367 | MAE : 0.3134 | R2 : 0.3464')
print(f'  Delta R2: {cnn_r2 - 0.3464:+.4f}')

In [ ]:
# ── 9. Loss curve ──────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(range(1, EPOCHS + 1), loss_history, color='#3fb950', linewidth=1.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title(f'HeatCNN Training Loss — {EPOCHS} epochs on {torch.cuda.get_device_name(0)}')
ax.set_facecolor('#161b22')
fig.patch.set_facecolor('#0d1117')
ax.tick_params(colors='#8b949e')
ax.spines[:].set_color('#30363d')
ax.title.set_color('#e6edf3')
ax.xaxis.label.set_color('#8b949e')
ax.yaxis.label.set_color('#8b949e')
ax.grid(alpha=0.15)
plt.tight_layout()
plt.savefig(DATA + 'cnn_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Loss curve saved to Drive.')

In [ ]:
# ── 10. Save model weights to Drive ───────────────────────────────────────────
import json

torch.save(cnn_model.state_dict(), DATA + 'cnn_model_100ep.pt')

# Save metrics so you can paste them back into the local notebook
metrics = {
    'epochs': EPOCHS,
    'device': torch.cuda.get_device_name(0),
    'rmse': round(float(cnn_rmse), 4),
    'mae':  round(float(cnn_mae), 4),
    'r2':   round(float(cnn_r2), 4),
}
with open(DATA + 'cnn_metrics_100ep.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print('Saved to Google Drive:')
print('  cnn_model_100ep.pt      -- model weights')
print('  cnn_metrics_100ep.json  -- paste these back into local notebook')
print()
print('Metrics to copy back:')
print(json.dumps(metrics, indent=2))